# 05 — Fairness Analysis

Цель ноутбука: проверить senior/junior share, концентрацию рекомендаций, среднюю загрузку рекомендованных сотрудников и баланс по ролям.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

In [ ]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
predictions = pd.read_csv(REPORTS / "test_predictions.csv")

scored = predictions.merge(employees, on="employee_id", how="left")
display(scored.head())

In [ ]:
top_by_task = (
    scored.sort_values(["task_id", "success_probability"], ascending=[True, False])
    .groupby("task_id")
    .head(1)
)

top_employee_share = top_by_task["employee_id"].value_counts(normalize=True)

fairness_summary = {
    "recommended_tasks": int(top_by_task["task_id"].nunique()),
    "senior_recommendation_share": float((top_by_task["grade"] == "senior").mean()),
    "junior_recommendation_share": float((top_by_task["grade"] == "junior").mean()),
    "top_employee_concentration": float(top_employee_share.head(1).iloc[0]),
    "average_recommended_workload": float(top_by_task["current_workload"].mean()),
}

display(fairness_summary)

In [ ]:
grade_counts = top_by_task["grade"].value_counts().reset_index()
grade_counts.columns = ["grade", "count"]

fig = px.bar(grade_counts, x="grade", y="count", title="Top recommendations by grade")
fig.show()
fig.write_html(FIGURES / "fairness_recommendations_by_grade.html")

In [ ]:
role_counts = top_by_task["role"].value_counts().reset_index()
role_counts.columns = ["role", "count"]

fig = px.bar(role_counts, x="role", y="count", title="Top recommendations by role")
fig.show()
fig.write_html(FIGURES / "fairness_recommendations_by_role.html")

In [ ]:
employee_concentration = top_by_task["name"].value_counts().head(10).reset_index()
employee_concentration.columns = ["name", "top1_recommendations"]

fig = px.bar(
    employee_concentration,
    x="name",
    y="top1_recommendations",
    title="Top employee concentration",
)
fig.show()
fig.write_html(FIGURES / "fairness_top_employee_concentration.html")

In [ ]:
fairness_path = REPORTS / "fairness_summary.json"

with open(fairness_path, "w", encoding="utf-8") as file:
    json.dump(fairness_summary, file, ensure_ascii=False, indent=2)

print("saved", fairness_path)